In [ ]:
import sys
from pathlib import Path

import pandas as pd

sys.path.append("..")
from modeling import COVARIATES, N_PERMUTATIONS, bh_fdr, oof_pearson, permutation_p, permuted

DATA_DIR = Path("/home/ec2-user/studies/Haochen_food/code_micro/data")
TRAIN_DIR = DATA_DIR / "train_pathways"

model_input = pd.read_csv(TRAIN_DIR / "model_input_diet_cov_functional_pathways_baseline.csv", low_memory=False)
results = pd.read_csv(TRAIN_DIR / "model_results_diet_predict_functional_pathways_baseline.csv")
summary = pd.read_csv(TRAIN_DIR / "functional_pathway_summary_baseline.csv").set_index("pathway")
pathways = summary.index[summary["kept_for_modeling"]].tolist()
features = COVARIATES + [c for c in model_input.columns if c not in ["participant_id", *COVARIATES, *summary.index]]
observed = results[results["model"] == "covariates_plus_diet"].set_index("pathway")["pearson_r"]

prevalence = summary.loc[pathways, "prevalence"] * 100
prevalence_bin = pd.cut(prevalence.clip(upper=99.999), bins=range(0, 105, 5), right=False, labels=range(0, 100, 5)).astype(int)
representative = {b: group.sort_values(kind="stable").index[len(group) // 2] for b, group in prevalence.groupby(prevalence_bin)}

In [ ]:
null = {}
for bin_start, rep in representative.items():
    y = model_input[rep].to_numpy(dtype=float)
    null[bin_start] = [oof_pearson(model_input[features], permuted(y, seed)) for seed in range(N_PERMUTATIONS)]

permutation = pd.DataFrame({"pathway": pathways, "observed_pearson_r": observed[pathways].to_numpy(),
                            "prevalence_percent": prevalence.to_numpy(), "prevalence_bin": prevalence_bin.to_numpy()})
permutation["perm_p_value"] = [permutation_p(null[b], r) for b, r in zip(permutation["prevalence_bin"], permutation["observed_pearson_r"])]
permutation["perm_q_value"] = bh_fdr(permutation["perm_p_value"])
permutation["perm_significant_fdr"] = permutation["perm_q_value"] < 0.05

out_dir = TRAIN_DIR / "permutation_test"
out_dir.mkdir(parents=True, exist_ok=True)
permutation.to_csv(out_dir / "functional_pathway_permutation_pvalues_fdr_baseline.csv", index=False)
pd.DataFrame(null).add_prefix("bin_").to_csv(out_dir / "functional_pathway_permutation_null_scores_baseline.csv", index=False)
print("significantly predicted pathways:", int(permutation["perm_significant_fdr"].sum()), "of", len(permutation))